<a href="https://colab.research.google.com/github/cyenseh-cmyk/bsr-llm-posttraining/blob/main/task01/bsr-llm-posttraining-task01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Behaviorally Sufficient Rank of LLM Post-Training Updates

This Google Colab notebook initializes the HDDA research project environment for the repository:

`cyenseh-cmyk/bsr-llm-posttraining`

The notebook is designed to:

1. Verify the Colab Python and GPU environment.
2. Clone a fresh copy of the GitHub repository into Colab.
3. Enter the repository.
4. Create the project directory structure.
5. Install the packages required for Task 01.
6. Verify that PyTorch, CUDA, Transformers, Accelerate, and Pandas work correctly.



## Step 1.1 — Check the Python Version

Task 01 requires Python 3.10 or newer.

Google Colab already provides Python, so this cell simply prints the version currently available in the runtime.

In [ ]:
!python --version

## Step 1.2 — Check PyTorch and GPU Availability

PyTorch is used throughout the project for tensors, neural-network weight matrices, GPU computation, and singular-value decomposition (SVD).

This cell checks:

- the installed PyTorch version,
- whether CUDA is available,
- the assigned GPU name,
- and the available GPU memory.

If `CUDA available` is `False`, enable a GPU runtime in Colab before continuing.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU memory: {total_gb:.2f} GB")
else:
    print("GPU is not enabled. In Colab, change the runtime to use a GPU before continuing.")

PyTorch version: 2.11.0+cpu
CUDA available: False
GPU is not enabled. In Colab, change the runtime to use a GPU before continuing.


## Step 1.3 — Prepare a Fresh Local Clone of the GitHub Repository

Colab uses `/content` as temporary working storage.

To make the setup reproducible, this cell removes any **existing local Colab copy** of `bsr-llm-posttraining` and then clones a fresh copy from GitHub.

This does **not** delete or overwrite the remote GitHub repository. It only refreshes the temporary copy inside the current Colab runtime.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = "https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git"
REPO_DIR = Path("/content/bsr-llm-posttraining")

if REPO_DIR.exists():
    print(f"Removing existing local Colab copy: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

print("Cloning repository...")
result = subprocess.run(
    ["git", "clone", REPO_URL, str(REPO_DIR)],
    text=True,
    capture_output=True
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "The repository could not be cloned. "
        "Confirm that cyenseh-cmyk/bsr-llm-posttraining exists and is accessible."
    )

print("Repository cloned successfully.")

Cloning repository...

Cloning into '/content/bsr-llm-posttraining'...

Repository cloned successfully.


## Step 1.4 — Enter the Project Repository

After cloning, we change the notebook's working directory to the repository.

All relative paths used in later cells will now be created inside:

`/content/bsr-llm-posttraining`

In [ ]:
%cd /content/bsr-llm-posttraining

/content/bsr-llm-posttraining


## Step 1.5 — Verify the Repository

This cell confirms:

- the current working directory,
- the Git remote URL,
- the active branch,
- and the current Git status.

This helps ensure that the Colab workspace is connected to the expected project repository.

In [ ]:
import os

print("Current directory:")
!pwd

print("\nGit remote:")
!git remote -v

print("\nCurrent branch:")
!git branch --show-current

print("\nGit status:")
!git status

Current directory:
/content/bsr-llm-posttraining

Git remote:
origin	https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git (fetch)
origin	https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git (push)

Current branch:
main

Git status:
On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


## Step 1.6 — Create the Project Directory Structure

The project is divided into clear folders so that data preparation, training, spectral analysis, reconstruction, evaluation, and results remain separate.

The main structure created here is:

- `task01/outputs/` — outputs required by Task 01
- `data/raw/` — original datasets or downloaded raw data
- `data/processed/` — reproducible processed datasets
- `training/sft/` — SFT-related scripts and configurations
- `training/grpo/` — GRPO-related scripts and configurations
- `checkpoints/sft/` and `checkpoints/grpo/` — model checkpoint locations
- `spectral_analysis/` — SVD, MP, OHT, and rank-analysis code
- `reconstruction/` — low-rank checkpoint reconstruction
- `evaluation/` — GSM8K and behavioral evaluation code
- `results/tables/` and `results/figures/` — final project outputs

Small `.gitkeep` files are added so that Git can preserve empty project directories.

In [ ]:
from pathlib import Path

PROJECT_DIRS = [
    "task01/outputs",
    "data/raw",
    "data/processed",
    "training/sft",
    "training/grpo",
    "checkpoints/sft",
    "checkpoints/grpo",
    "spectral_analysis",
    "reconstruction",
    "evaluation",
    "results/tables",
    "results/figures",
]

for folder in PROJECT_DIRS:
    path = Path(folder)
    path.mkdir(parents=True, exist_ok=True)
    (path / ".gitkeep").touch(exist_ok=True)

print("Project folders created successfully.\n")
for folder in PROJECT_DIRS:
    print(folder)

Project folders created successfully.

task01/outputs
data/raw
data/processed
training/sft
training/grpo
checkpoints/sft
checkpoints/grpo
spectral_analysis
reconstruction
evaluation
results/tables
results/figures


## Step 1.7 — Create or Update `.gitignore`

Large model checkpoints and temporary Python files should not be committed to GitHub.

This cell creates a `.gitignore` that excludes:

- Python cache files,
- notebook checkpoints,
- local virtual environments,
- Hugging Face caches,
- experiment logs,
- and large model-weight file formats.

The project folders themselves remain available, but large generated model files are kept out of Git.

In [ ]:
gitignore = r'''
# Python
__pycache__/
*.py[cod]
*.pyo
.ipynb_checkpoints/
.venv/
venv/

# OS / editor
.DS_Store
Thumbs.db
.vscode/

# Hugging Face / caches
.cache/
huggingface/
hf_cache/

# Experiment logs
wandb/
runs/

# Large model/checkpoint files
*.pt
*.pth
*.ckpt
*.bin
*.safetensors

# Temporary files
*.tmp
*.log
'''.lstrip()

Path(".gitignore").write_text(gitignore, encoding="utf-8")
print(Path(".gitignore").read_text())

# Python
__pycache__/
*.py[cod]
*.pyo
.ipynb_checkpoints/
.venv/
venv/

# OS / editor
.DS_Store
Thumbs.db
.vscode/

# Hugging Face / caches
.cache/
huggingface/
hf_cache/

# Experiment logs
wandb/
runs/

# Large model/checkpoint files
*.pt
*.pth
*.ckpt
*.bin
*.safetensors

# Temporary files
*.tmp
*.log



## Step 1.8 — Install the Packages Required for Task 01

The Task 01 specification requires:

- `torch`
- `transformers`
- `accelerate`
- `pandas`

Colab already provides a GPU-compatible PyTorch installation, so this cell does not reinstall PyTorch. Reinstalling PyTorch unnecessarily can break the CUDA configuration supplied by Colab.

The remaining required libraries are installed here.

In [ ]:
!pip install -q transformers accelerate pandas

## Step 1.9 — Verify the Required Libraries

This cell imports all required libraries and prints their versions.

A successful run with no import errors confirms that the main Task 01 software dependencies are available.

In [ ]:
import torch
import transformers
import accelerate
import pandas as pd

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Pandas:", pd.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Transformers: 5.18.0
Accelerate: 1.15.0
Pandas: 2.2.3

CUDA available: True
GPU: Tesla T4


## Step 1.10 — Verify Basic PyTorch Matrix Operations

The project studies neural-network weight matrices, so we perform a small matrix test before loading Qwen.

This cell creates a 2 × 2 tensor and prints its shape.

In [ ]:
test_matrix = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

print(test_matrix)
print("Shape:", test_matrix.shape)

tensor([[1., 2.],
        [3., 4.]])
Shape: torch.Size([2, 2])


## Step 1.11 — Verify GPU Tensor Operations

Detecting CUDA is not enough; we also want to confirm that PyTorch can actually move a tensor to the GPU.

If CUDA is available, this cell transfers the test matrix to `cuda:0`.

In [ ]:
if torch.cuda.is_available():
    test_matrix_gpu = test_matrix.to("cuda")
    print("Tensor device:", test_matrix_gpu.device)
else:
    print("CUDA is not available. The tensor remains on CPU.")

Tensor device: cuda:0


## Step 1.12 — Verify Singular-Value Decomposition

Task 01 later requires `torch.linalg.svdvals(W)` for every selected Transformer weight matrix.

This small test confirms that SVD works correctly before we apply it to Qwen's much larger matrices.

In [ ]:
W = test_matrix.to(torch.float32)
singular_values = torch.linalg.svdvals(W)

print("Singular values:", singular_values)
print("Largest singular value:", singular_values[0].item())

Singular values: tensor([5.4650, 0.3660])
Largest singular value: 5.4649858474731445


## Step 1.13 — Verify Pandas Table Operations

Pandas will be used to create:

- `matrix_inventory.csv`
- `spectral_summary.csv`

This cell creates a small example table to confirm that Pandas is functioning correctly.

In [ ]:
test_df = pd.DataFrame({
    "layer": [0, 1],
    "matrix_family": ["q_proj", "k_proj"]
})

test_df

,layer,matrix_family
0,0,q_proj
1,1,k_proj


## Step 1.14 — Verify the Final Project Structure

This cell prints the project directories that were created during setup.

At this point, the Colab environment should be ready for **Task 01 Part A: loading and inspecting `PrimeIntellect/Qwen3-0.6B`**.

In [ ]:
from pathlib import Path

for path in sorted(Path(".").rglob("*")):
    if path.is_dir() and ".git" not in path.parts:
        print(path)

checkpoints
checkpoints/grpo
checkpoints/sft
data
data/processed
data/raw
evaluation
reconstruction
results
results/figures
results/tables
spectral_analysis
task01
task01/outputs
training
training/grpo
training/sft


## Step 1.7 — Verify the Git Repository

The project folders have now been created.

Before installing the remaining software dependencies, we verify that the current Colab workspace is connected to the correct GitHub repository and that Git recognizes the project correctly.

This cell displays:

- the current working directory,
- the configured GitHub remote,
- the active Git branch,
- and the current repository status.

In [ ]:
print("Current directory:")
!pwd

print("\nGit remote:")
!git remote -v

print("\nCurrent branch:")
!git branch --show-current

print("\nGit status:")
!git status

Current directory:
/content/bsr-llm-posttraining

Git remote:
origin	https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git (fetch)
origin	https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git (push)

Current branch:
main

Git status:
On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	checkpoints/
	data/
	evaluation/
	reconstruction/
	results/
	spectral_analysis/
	task01/
	training/

nothing added to commit but untracked files present (use "git add" to track)


## Step 1.8 — Install the Required Python Libraries

Task 01 requires several Python libraries.

### PyTorch

PyTorch is used for tensors, neural-network weight matrices, GPU computation, and Singular Value Decomposition (SVD).

Google Colab already provides PyTorch, so we do not reinstall it.

### Transformers

The Hugging Face `transformers` library will be used to load the Qwen3-0.6B tokenizer and language model.

### Accelerate

`accelerate` helps Hugging Face manage model loading and available hardware such as CPU and GPU.

### Pandas

`pandas` will be used to organize model information into tables and save results such as `matrix_inventory.csv` and `spectral_summary.csv`.

The following cell installs the packages that are not guaranteed to be available in the Colab environment.

In [ ]:
!pip install -q transformers accelerate pandas

In [ ]:
import torch
import transformers
import accelerate
import pandas as pd

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Pandas:", pd.__version__)

PyTorch: 2.11.0+cpu
Transformers: 5.18.0
Accelerate: 1.15.0
Pandas: 2.2.3


In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected.")

CUDA available: False
No GPU detected.


In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!nvidia-smi

print("\n--- PyTorch CUDA Check ---")

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is still unavailable.")

Wed Oct  7 13:00:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git

Cloning into 'bsr-llm-posttraining'...


In [ ]:
%cd /content/bsr-llm-posttraining


/content/bsr-llm-posttraining


In [ ]:
!pwd
!git remote -v
!git branch --show-current

/content/bsr-llm-posttraining
origin	https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git (fetch)
origin	https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git (push)
main


In [ ]:
!git status

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	checkpoints/
	data/
	evaluation/
	reconstruction/
	results/
	spectral_analysis/
	task01/
	training/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git add .

In [ ]:
!git config user.name "Curtis Kuti Yenseh"
!git config user.email "cyenseh@gmail.com"


In [ ]:
print("Git user name:")
!git config user.name

print("\nGit user email:")
!git config user.email

Git user name:
Curtis Kuti Yenseh

Git user email:
cyenseh@gmail.com


In [ ]:
!git commit -m "Initialize Colab project structure and environment setup"

[main (root-commit) fdb1269] Initialize Colab project structure and environment setup
 13 files changed, 32 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 checkpoints/grpo/.gitkeep
 create mode 100644 checkpoints/sft/.gitkeep
 create mode 100644 data/processed/.gitkeep
 create mode 100644 data/raw/.gitkeep
 create mode 100644 evaluation/.gitkeep
 create mode 100644 reconstruction/.gitkeep
 create mode 100644 results/figures/.gitkeep
 create mode 100644 results/tables/.gitkeep
 create mode 100644 spectral_analysis/.gitkeep
 create mode 100644 task01/outputs/.gitkeep
 create mode 100644 training/grpo/.gitkeep
 create mode 100644 training/sft/.gitkeep


In [ ]:
!git status

On branch main
Your branch is based on 'origin/main', but the upstream is gone.
  (use "git branch --unset-upstream" to fixup)

nothing to commit, working tree clean


In [ ]:
!git config user.name "Yenseh Curtis Kuti"
!git config user.email "cyenseh@gmail.com"

In [ ]:
print("Git user name:")
!git config user.name

print("\nGit user email:")
!git config user.email


Git user name:
Yenseh Curtis Kuti

Git user email:
cyenseh@gmail.com


In [ ]:
!git commit -m "Initialize Colab project structure and environment setup"

On branch main
Your branch is based on 'origin/main', but the upstream is gone.
  (use "git branch --unset-upstream" to fixup)

nothing to commit, working tree clean


In [ ]:
from getpass import getpass
import os
import subprocess
import tempfile

GITHUB_USER = "cyenseh-cmyk"

# Enter the token securely.
GITHUB_TOKEN = getpass("Enter your GitHub Personal Access Token: ")

askpass_script = """#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USER" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""

with tempfile.NamedTemporaryFile(
    mode="w",
    delete=False,
    suffix=".sh"
) as f:
    f.write(askpass_script)
    askpass_path = f.name

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = askpass_path
env["GIT_TERMINAL_PROMPT"] = "0"
env["GITHUB_USER"] = GITHUB_USER
env["GITHUB_TOKEN"] = GITHUB_TOKEN

result = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

os.remove(askpass_path)
GITHUB_TOKEN = None

Enter your GitHub Personal Access Token: ··········
branch 'main' set up to track 'origin/main'.

To https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git
 * [new branch]      main -> main



In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [1]:
import os
import sys
import torch

print("Current directory:", os.getcwd())
print("\nProject files/folders:")
print(os.listdir("."))

print("\nPython:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

packages = [
    "transformers",
    "datasets",
    "accelerate"
]

print("\nPackage check:")
for package in packages:
    try:
        module = __import__(package)
        print(f"{package}: OK")
    except ImportError:
        print(f"{package}: NOT INSTALLED")

Current directory: /content

Project files/folders:
['.config', 'sample_data']

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4

Package check:
transformers: OK
datasets: OK
accelerate: OK


In [2]:
!git clone https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git
%cd /content/bsr-llm-posttraining

Cloning into 'bsr-llm-posttraining'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 9 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), done.
/content/bsr-llm-posttraining


In [3]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [4]:
import os

print("Current directory:", os.getcwd())
print("\nProject contents:")
print(os.listdir("."))

Current directory: /content/bsr-llm-posttraining

Project contents:
['.git', 'spectral_analysis', 'data', 'reconstruction', 'training', 'evaluation', 'checkpoints', 'task01', '.gitignore', 'results']


## Step 2: Load the Qwen3-0.6B Model

In this step, we load the original language model that will be used throughout the experiment.

The model used in this project is **Qwen3-0.6B**.

This original model represents our starting point, which we can think of mathematically as:

\[
W_0
\]

where \(W_0\) represents the model weights before any additional training.

Later, we will train this same model using SFT and GRPO and compare the trained model weights with these original weights.

### What this code does

The code below performs four main tasks:

1. **Imports PyTorch and Hugging Face Transformers**
   - `torch` is used to work with neural networks and the GPU.
   - `AutoTokenizer` is used to load the tokenizer for Qwen.
   - `AutoModelForCausalLM` is used to load the actual Qwen language model.

2. **Defines the model name**
   - We specify `Qwen/Qwen3-0.6B` as the model we want to use.

3. **Loads the tokenizer**
   - A language model cannot directly understand normal text.
   - The tokenizer converts text into numbers called **tokens** that the model can process.

For example:

`"What is 2 + 2?"`

is converted into a sequence of numerical tokens before being passed to Qwen.

4. **Loads the Qwen model onto the GPU**
   - The model is loaded using 16-bit floating-point numbers (`float16`) to reduce GPU memory usage.
   - `device_map="auto"` allows Hugging Face to automatically place the model on the available GPU.

Finally, the code prints the model name, device, and numerical data type so that we can confirm that everything loaded correctly.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-0.6B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\nModel loaded successfully!")
print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Data type:", next(model.parameters()).dtype)

Loading tokenizer...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Model loaded successfully!
Model: Qwen/Qwen3-0.6B
Device: cuda:0
Data type: torch.float16


### Why this step is important

This model is the **baseline model** for the entire project.

At this stage, we have not performed SFT or GRPO training yet.

Therefore, the model weights currently represent:

\[
W_0
\]

Later, after training, we will obtain new model weights:

\[
W_t
\]

and calculate the training update:

\[
\Delta W = W_t - W_0
\]

The matrix \(\Delta W\) tells us exactly how the model changed during post-training.

These weight changes will later be analyzed using Singular Value Decomposition (SVD), Marchenko–Pastur, Optimal Hard Thresholding, and the 95% explained-variance baseline.

## Step 3: Test the Loaded Qwen Model

Before using Qwen3-0.6B for the research experiment, we first perform a simple generation test.

The purpose of this step is to confirm that:

- the tokenizer works correctly,
- the model receives input correctly,
- the GPU can perform inference,
- and Qwen can generate a response.

This is only a functionality test. It is not part of the final GSM8K evaluation.

In [6]:
prompt = "What is 2 + 2?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Prompt:")
print(prompt)

print("\nQwen response:")
print(response)

Prompt:
What is 2 + 2?

Qwen response:
What is 2 + 2? What is 2 + 2 + 2? What is 2 + 2 + 2 + 2? What is 2 + 2 + 2 + 2 + 2? What is 2 + 2 +


### Correcting the Qwen Generation Format

The previous test confirmed that the model could generate text, but the output repeatedly copied the prompt instead of answering the question.

This happened because Qwen3 is designed to receive prompts in a structured chat format.

In this corrected test, we:

1. Represent the question as a user message.
2. Apply Qwen's official chat template.
3. Disable thinking mode for this simple functionality test.
4. Generate the model's response.
5. Decode only the newly generated tokens instead of decoding the original prompt together with the answer.

This gives us a cleaner test of whether Qwen can respond correctly.

In [7]:
prompt = "What is 2 + 2?"

messages = [
    {"role": "user", "content": prompt}
]

# Apply Qwen's chat format
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# Convert the formatted text into tokens and move them to the GPU
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# Generate the answer
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.8
    )

# Keep only the newly generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Prompt:")
print(prompt)

print("\nQwen response:")
print(response)

Prompt:
What is 2 + 2?

Qwen response:
2 + 2 = 4.


## Step 4: Load and Inspect the GSM8K Dataset

In this step, we load the GSM8K dataset, which is the mathematics dataset used in this project.

GSM8K contains grade-school mathematics word problems together with worked solutions.

This dataset is important because:

- Qwen will be post-trained on GSM8K problems.
- SFT will learn from the provided solutions.
- GRPO will use the same problems with a reward based on whether the final answer is correct.
- Final model performance will be evaluated on GSM8K accuracy.

Before creating our experimental splits, we first load the dataset and inspect its structure.

### What this code does

The code below:

1. Imports the Hugging Face `datasets` library.
2. Downloads the GSM8K dataset.
3. Loads the standard `main` configuration.
4. Displays the available dataset splits.
5. Prints the number of examples in each split.
6. Shows one example so that we can understand the structure of the data.

At this stage, we are only inspecting the dataset. We are not training the model yet.

In [8]:
from datasets import load_dataset

print("Loading GSM8K dataset...")

gsm8k = load_dataset(
    "openai/gsm8k",
    "main"
)

print("\nDataset loaded successfully!")
print(gsm8k)

print("\nAvailable splits:")
for split_name in gsm8k.keys():
    print(f"{split_name}: {len(gsm8k[split_name])} examples")

print("\nExample from the training set:")
example = gsm8k["train"][0]

print("\nQuestion:")
print(example["question"])

print("\nAnswer:")
print(example["answer"])


Loading GSM8K dataset...


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

Available splits:
train: 7473 examples
test: 1319 examples

Example from the training set:

Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Answer:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


## Step 5: Create the Fixed GSM8K Experimental Splits

The research proposal specifies that the GSM8K data should be divided as follows:

- **1,024 problems** for post-training the model using SFT and GRPO.
- **256 different problems** reserved for the rank-sweep and reconstruction experiments.
- The official **GSM8K test split** is kept untouched for final evaluation.

It is important that the 1,024 training problems and the 256 rank-sweep problems do not overlap.

To make the experiment reproducible, we use a fixed random seed. This ensures that every time the experiment is reproduced, exactly the same GSM8K examples are selected.

### Why do we need a fixed random seed?

The GSM8K training set contains 7,473 examples, but our project only uses a fixed subset.

If we randomly selected examples without controlling the randomness, we could obtain a different subset every time the notebook is run.

We therefore choose:

`SEED = 42`

The number 42 itself is not mathematically special. It is simply a fixed value that allows us to reproduce exactly the same split later.

The split will be:

1. Shuffle the original GSM8K training set using seed 42.
2. Take the first 1,024 examples as the post-training set.
3. Take the next 256 examples as the rank-sweep set.
4. Leave the official GSM8K test set unchanged.

In [9]:
import json
import os

SEED = 42
TRAIN_SIZE = 1024
RANK_SWEEP_SIZE = 256

# Add the original index of every GSM8K training example.
# This allows us to save exactly which examples were selected.
gsm8k_train_with_ids = gsm8k["train"].add_column(
    "source_index",
    list(range(len(gsm8k["train"])))
)

# Shuffle reproducibly
shuffled_train = gsm8k_train_with_ids.shuffle(seed=SEED)

# Select the 1,024 post-training examples
train_dataset = shuffled_train.select(
    range(TRAIN_SIZE)
)

# Select a separate 256 examples for the rank sweep
rank_sweep_dataset = shuffled_train.select(
    range(TRAIN_SIZE, TRAIN_SIZE + RANK_SWEEP_SIZE)
)

# Keep the official GSM8K test split untouched
test_dataset = gsm8k["test"]

print("Experimental splits created successfully!")

print("\nPost-training set:")
print(len(train_dataset), "examples")

print("\nRank-sweep set:")
print(len(rank_sweep_dataset), "examples")

print("\nOfficial test set:")
print(len(test_dataset), "examples")

Experimental splits created successfully!

Post-training set:
1024 examples

Rank-sweep set:
256 examples

Official test set:
1319 examples


### Verify That the Training and Rank-Sweep Sets Do Not Overlap

The rank-sweep set must contain different GSM8K problems from the post-training set.

If the same examples appeared in both sets, our evaluation could become biased because the model would already have trained on problems that we later use to decide the appropriate rank.

We therefore compare the original GSM8K indices of both subsets and verify that their intersection is empty.

In [10]:
train_ids = set(train_dataset["source_index"])
rank_ids = set(rank_sweep_dataset["source_index"])

overlap = train_ids.intersection(rank_ids)

print("Number of overlapping examples:", len(overlap))

if len(overlap) == 0:
    print("Success: the two datasets are completely separate.")
else:
    print("Warning: overlapping examples were found.")

Number of overlapping examples: 0
Success: the two datasets are completely separate.


### Save the Experimental Split Information

To make the experiment fully reproducible, we save the original GSM8K indices used for the two custom subsets.

This means that even if the Colab runtime is reset, we can reload GSM8K and reconstruct exactly the same:

- 1,024 post-training examples
- 256 rank-sweep examples

without relying on a new random selection.

The official GSM8K test set does not require custom indices because it is used in its original form.

In [12]:
import json
import os

split_info = {
    "seed": SEED,
    "train_size": TRAIN_SIZE,
    "rank_sweep_size": RANK_SWEEP_SIZE,
    "train_indices": list(train_dataset["source_index"]),
    "rank_sweep_indices": list(rank_sweep_dataset["source_index"])
}

split_path = "/content/bsr-llm-posttraining/data/gsm8k_split_indices.json"

with open(split_path, "w") as f:
    json.dump(split_info, f, indent=2)

print("Split information saved successfully!")
print("Location:")
print(split_path)

Split information saved successfully!
Location:
/content/bsr-llm-posttraining/data/gsm8k_split_indices.json


In [13]:
with open(split_path, "r") as f:
    saved_split = json.load(f)

print("Seed:", saved_split["seed"])
print("Training examples:", len(saved_split["train_indices"]))
print("Rank-sweep examples:", len(saved_split["rank_sweep_indices"]))

print("\nFirst 10 training indices:")
print(saved_split["train_indices"][:10])

print("\nFirst 10 rank-sweep indices:")
print(saved_split["rank_sweep_indices"][:10])

Seed: 42
Training examples: 1024
Rank-sweep examples: 256

First 10 training indices:
[3082, 2184, 5897, 2437, 6330, 819, 3812, 5878, 5302, 5876]

First 10 rank-sweep indices:
[500, 3202, 2764, 6414, 4495, 502, 4195, 1826, 5020, 2166]


In [14]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/gsm8k_split_indices.json

nothing added to commit but untracked files present (use "git add" to track)


In [15]:
!git add data/gsm8k_split_indices.json

In [17]:
!git config --global user.name "Yenseh Curtis Kuti"
!git config --global user.email "cyenseh@gmail.com"

In [18]:
!git config --global user.name
!git config --global user.email

Yenseh Curtis Kuti
cyenseh@gmail.com


In [19]:
!git commit -m "Add reproducible GSM8K experimental split"

[main 0d7cfef] Add reproducible GSM8K experimental split
 1 file changed, 1289 insertions(+)
 create mode 100644 data/gsm8k_split_indices.json


In [20]:
!git status


On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [22]:
import os
import getpass
import subprocess
import tempfile

GITHUB_USERNAME = "cyenseh-cmyk"

# Enter the token securely.
# The token will NOT be displayed while typing.
github_token = getpass.getpass(
    "Enter your GitHub Personal Access Token: "
)

# Create a temporary Git askpass script
askpass_path = "/tmp/git_askpass.sh"

with open(askpass_path, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USERNAME" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

os.chmod(askpass_path, 0o700)

# Temporary authentication environment
env = os.environ.copy()
env["GIT_ASKPASS"] = askpass_path
env["GITHUB_USERNAME"] = GITHUB_USERNAME
env["GITHUB_TOKEN"] = github_token
env["GIT_TERMINAL_PROMPT"] = "0"

# Push the existing commit
result = subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

# Remove temporary credentials
os.remove(askpass_path)
github_token = None


Enter your GitHub Personal Access Token: ··········

To https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git
   fdb1269..0d7cfef  main -> main



In [23]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


### Step 7.1: Extract the Correct Answer from GSM8K

Each GSM8K solution contains a worked explanation followed by the final numerical answer.

The final answer appears after:

`####`

For example:

`#### 72`

For evaluation, we do not need to compare Qwen's full explanation with the reference explanation.

We only need to compare the final numerical answers.

The function below extracts the correct final answer from each GSM8K reference solution.

In [24]:
import re

def extract_gsm8k_gold_answer(answer_text):
    """
    Extract the final numerical answer appearing after ####
    in a GSM8K reference solution.
    """

    match = re.search(
        r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        answer_text
    )

    if match:
        return match.group(1).replace(",", "")

    return None

In [25]:
sample_answer = gsm8k["train"][0]["answer"]

print("Original GSM8K answer:")
print(sample_answer)

print("\nExtracted final answer:")
print(extract_gsm8k_gold_answer(sample_answer))

Original GSM8K answer:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

Extracted final answer:
72


### Step 7.2: Test Qwen on Five GSM8K Problems

Before evaluating the original Qwen model on the complete GSM8K test set, we first test the evaluation pipeline on only five problems.

For each problem, we will:

1. Take a question from the GSM8K test set.
2. Ask Qwen to solve it.
3. Ask Qwen to place its final numerical answer after `####`.
4. Extract Qwen's final answer.
5. Extract the correct GSM8K answer.
6. Compare the two answers.
7. Mark the prediction as correct or incorrect.

This small test allows us to detect problems with prompting, generation, or answer extraction before running a much larger evaluation.

The results from these five examples are only a pipeline check and will not be used as the final baseline accuracy.

In [26]:
def extract_model_answer(response_text):
    """
    Extract Qwen's final numerical answer.

    First look for the preferred GSM8K format:
        #### number

    If that is not found, use the last number appearing
    in the generated response.
    """

    # Preferred format
    match = re.search(
        r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        response_text
    )

    if match:
        return match.group(1).replace(",", "")

    # Fallback: take the last numerical value
    numbers = re.findall(
        r"[-+]?\d[\d,]*(?:\.\d+)?",
        response_text
    )

    if numbers:
        return numbers[-1].replace(",", "")

    return None

In [27]:
def generate_gsm8k_answer(question):
    """
    Ask Qwen to solve one GSM8K problem.
    """

    messages = [
        {
            "role": "user",
            "content": (
                "Solve the following grade-school math problem. "
                "Show your reasoning clearly. "
                "At the end, write the final numerical answer exactly "
                "in the format: #### answer\n\n"
                f"Problem:\n{question}"
            )
        }
    ]

    # Apply Qwen's chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Generate deterministically
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Keep only newly generated tokens
    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

#### Run the Five-Question Evaluation

We now evaluate Qwen on the first five examples from the official GSM8K test split.

For each question, the notebook displays:

- the question,
- the correct answer,
- Qwen's response,
- Qwen's extracted final answer,
- and whether the answer was correct.

At the end, we calculate a temporary accuracy over the five examples.

This accuracy is only used to verify that the evaluation pipeline works correctly.

In [28]:
NUM_TEST_EXAMPLES = 5

correct = 0

for i in range(NUM_TEST_EXAMPLES):

    example = test_dataset[i]

    question = example["question"]

    gold_answer = extract_gsm8k_gold_answer(
        example["answer"]
    )

    response = generate_gsm8k_answer(question)

    predicted_answer = extract_model_answer(
        response
    )

    is_correct = predicted_answer == gold_answer

    if is_correct:
        correct += 1

    print("=" * 80)
    print(f"Example {i + 1}")
    print("=" * 80)

    print("\nQuestion:")
    print(question)

    print("\nGold answer:")
    print(gold_answer)

    print("\nQwen response:")
    print(response)

    print("\nExtracted Qwen answer:")
    print(predicted_answer)

    print("\nCorrect:")
    print(is_correct)

temporary_accuracy = correct / NUM_TEST_EXAMPLES

print("\n" + "=" * 80)
print("FIVE-QUESTION TEST SUMMARY")
print("=" * 80)

print(f"Correct answers: {correct}/{NUM_TEST_EXAMPLES}")
print(f"Temporary accuracy: {temporary_accuracy:.2%}")

Example 1

Question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Gold answer:
18

Qwen response:
To solve the problem, let's break it down step by step:

1. **Janet's ducks lay 16 eggs per day.**
2. **She eats 3 eggs for breakfast every morning.**
3. **She bakes 4 eggs for her friends every day.**
4. **The remainder is sold at the farmers' market for $2 per egg.**

### Step 1: Calculate the number of eggs eaten each day
- She eats 3 eggs for breakfast.
- She also eats 4 eggs for baking.
- So, she eats $ 3 + 4 = 7 $ eggs per day.

### Step 2: Calculate the number of eggs left each day
- Total eggs per day: 16
- Eggs eaten: 7
- Eggs left: $ 16 - 7 = 9 $

### Step 3: Calculate the money made from selling the eggs
- Each egg is sold for $2
- So, money made

## Task01 — Step 3: Inspect the Qwen3-0.6B Transformer Structure

Now that Qwen3-0.6B has been loaded successfully, the next step is to inspect the internal structure of the model.

The goal is to understand how the Transformer is organized before performing any spectral analysis.

In particular, we want to identify:

- the embedding layer,
- the Transformer blocks,
- the attention modules,
- the MLP modules,
- and the weight matrices inside each block.

This inspection will help us determine exactly which matrices should later be included in the singular-value analysis.

In [29]:
print(model)


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

### Count the Transformer Blocks

A Transformer model is built from several repeated Transformer blocks.

Each block contains components such as:

- self-attention,
- query, key, value and output projections,
- an MLP/feed-forward network,
- and normalization layers.

We first count how many Transformer blocks exist in Qwen3-0.6B.

In [30]:
num_layers = len(model.model.layers)

print("Number of Transformer blocks:", num_layers)

Number of Transformer blocks: 28


### Inspect Transformer Block 0

Instead of studying every Transformer block at once, we first inspect the first block, called **block 0**.

Because the Transformer blocks have similar structures, understanding one block helps us understand what kinds of matrices occur throughout the model.

We will later repeat the matrix inventory across every block.

In [31]:
block0 = model.model.layers[0]

print(block0)

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


## Task01 — Step 4: Build the Transformer Matrix Inventory

We have inspected Transformer Block 0 and identified the main projection matrices inside the attention and MLP modules.

The next step is to automatically inspect every Transformer block in Qwen3-0.6B and create an inventory of the projection weight matrices.

For each matrix, we will record:

- Transformer block number,
- matrix type,
- full parameter name,
- number of rows,
- number of columns,
- total number of parameters,
- and data type.

The seven projection matrices of interest are:

- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`
- `gate_proj`
- `up_proj`
- `down_proj`

This inventory will tell us exactly which matrices will later be used for spectral analysis.

In [32]:
import pandas as pd

target_matrices = {
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
}

matrix_records = []

for layer_idx, layer in enumerate(model.model.layers):

    # Attention matrices
    attention_modules = {
        "q_proj": layer.self_attn.q_proj,
        "k_proj": layer.self_attn.k_proj,
        "v_proj": layer.self_attn.v_proj,
        "o_proj": layer.self_attn.o_proj,
    }

    # MLP matrices
    mlp_modules = {
        "gate_proj": layer.mlp.gate_proj,
        "up_proj": layer.mlp.up_proj,
        "down_proj": layer.mlp.down_proj,
    }

    all_modules = {
        **attention_modules,
        **mlp_modules
    }

    for matrix_name, module in all_modules.items():

        weight = module.weight

        rows, cols = weight.shape

        matrix_records.append({
            "layer": layer_idx,
            "matrix": matrix_name,
            "rows": rows,
            "columns": cols,
            "parameters": weight.numel(),
            "dtype": str(weight.dtype)
        })


matrix_inventory = pd.DataFrame(matrix_records)

print("Matrix inventory created successfully!")
print()
print(matrix_inventory.head(14))

Matrix inventory created successfully!

    layer     matrix  rows  columns  parameters          dtype
0       0     q_proj  2048     1024     2097152  torch.float16
1       0     k_proj  1024     1024     1048576  torch.float16
2       0     v_proj  1024     1024     1048576  torch.float16
3       0     o_proj  1024     2048     2097152  torch.float16
4       0  gate_proj  3072     1024     3145728  torch.float16
5       0    up_proj  3072     1024     3145728  torch.float16
6       0  down_proj  1024     3072     3145728  torch.float16
7       1     q_proj  2048     1024     2097152  torch.float16
8       1     k_proj  1024     1024     1048576  torch.float16
9       1     v_proj  1024     1024     1048576  torch.float16
10      1     o_proj  1024     2048     2097152  torch.float16
11      1  gate_proj  3072     1024     3145728  torch.float16
12      1    up_proj  3072     1024     3145728  torch.float16
13      1  down_proj  1024     3072     3145728  torch.float16


## Task01 — Step 5: Save the Model Structure

The first Task01 output file is:

`task01/outputs/model_structure.txt`

This file records the architecture of Qwen3-0.6B so that we have a permanent description of the model that was inspected.

The file will contain:

- model name,
- model class,
- total number of parameters,
- number of Transformer blocks,
- complete model architecture,
- and the structure of Transformer Block 0.

Saving this information makes the model-inspection stage reproducible.

In [33]:
import os

OUTPUT_DIR = "/content/bsr-llm-posttraining/task01/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model_structure_path = os.path.join(
    OUTPUT_DIR,
    "model_structure.txt"
)

total_parameters = sum(
    p.numel() for p in model.parameters()
)

num_layers = len(model.model.layers)

with open(model_structure_path, "w") as f:

    f.write("TASK01 — QWEN3-0.6B MODEL STRUCTURE\n")
    f.write("=" * 60 + "\n\n")

    f.write(f"Model name: {MODEL_NAME}\n")
    f.write(f"Model class: {model.__class__.__name__}\n")
    f.write(f"Total parameters: {total_parameters:,}\n")
    f.write(f"Transformer blocks: {num_layers}\n")

    f.write("\n" + "=" * 60 + "\n")
    f.write("FULL MODEL ARCHITECTURE\n")
    f.write("=" * 60 + "\n\n")

    f.write(str(model))

    f.write("\n\n" + "=" * 60 + "\n")
    f.write("TRANSFORMER BLOCK 0\n")
    f.write("=" * 60 + "\n\n")

    f.write(str(block0))

print("Model structure saved successfully!")
print(model_structure_path)

Model structure saved successfully!
/content/bsr-llm-posttraining/task01/outputs/model_structure.txt


## Task01 — Step 6: Build and Save the Final Matrix Inventory

The second Task01 output file is:

`task01/outputs/matrix_inventory.csv`

For every target projection matrix in every Transformer block, we record:

- Transformer block number,
- matrix name,
- matrix group (`attention` or `mlp`),
- full parameter name,
- number of rows,
- number of columns,
- total number of parameters,
- and data type.

The seven target matrices are:

- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`
- `gate_proj`
- `up_proj`
- `down_proj`

This inventory defines the exact set of matrices that will later undergo spectral analysis.

In [34]:
import pandas as pd

matrix_records = []

for layer_idx, layer in enumerate(model.model.layers):

    modules = {
        "q_proj": (
            "attention",
            layer.self_attn.q_proj
        ),
        "k_proj": (
            "attention",
            layer.self_attn.k_proj
        ),
        "v_proj": (
            "attention",
            layer.self_attn.v_proj
        ),
        "o_proj": (
            "attention",
            layer.self_attn.o_proj
        ),
        "gate_proj": (
            "mlp",
            layer.mlp.gate_proj
        ),
        "up_proj": (
            "mlp",
            layer.mlp.up_proj
        ),
        "down_proj": (
            "mlp",
            layer.mlp.down_proj
        ),
    }

    for matrix_name, (group, module) in modules.items():

        weight = module.weight
        rows, cols = weight.shape

        if group == "attention":
            parameter_name = (
                f"model.layers.{layer_idx}."
                f"self_attn.{matrix_name}.weight"
            )
        else:
            parameter_name = (
                f"model.layers.{layer_idx}."
                f"mlp.{matrix_name}.weight"
            )

        matrix_records.append({
            "layer": layer_idx,
            "group": group,
            "matrix": matrix_name,
            "parameter_name": parameter_name,
            "rows": rows,
            "columns": cols,
            "parameters": weight.numel(),
            "dtype": str(weight.dtype)
        })

matrix_inventory = pd.DataFrame(matrix_records)

print("Final matrix inventory created!")
print()
print(matrix_inventory.head(14))

Final matrix inventory created!

    layer      group     matrix                          parameter_name  rows  \
0       0  attention     q_proj  model.layers.0.self_attn.q_proj.weight  2048   
1       0  attention     k_proj  model.layers.0.self_attn.k_proj.weight  1024   
2       0  attention     v_proj  model.layers.0.self_attn.v_proj.weight  1024   
3       0  attention     o_proj  model.layers.0.self_attn.o_proj.weight  1024   
4       0        mlp  gate_proj     model.layers.0.mlp.gate_proj.weight  3072   
5       0        mlp    up_proj       model.layers.0.mlp.up_proj.weight  3072   
6       0        mlp  down_proj     model.layers.0.mlp.down_proj.weight  1024   
7       1  attention     q_proj  model.layers.1.self_attn.q_proj.weight  2048   
8       1  attention     k_proj  model.layers.1.self_attn.k_proj.weight  1024   
9       1  attention     v_proj  model.layers.1.self_attn.v_proj.weight  1024   
10      1  attention     o_proj  model.layers.1.self_attn.o_proj.weight  102

### Validate the Matrix Inventory

Each Transformer block should contribute exactly seven projection matrices.

Therefore, if the model contains \(L\) Transformer blocks, the expected number of matrices is:

\[
7L
\]

We compare this expected number with the number actually found by our inventory code.

This check helps ensure that no target matrices were accidentally omitted.

In [35]:
expected_matrices = num_layers * 7
found_matrices = len(matrix_inventory)

print("Transformer blocks:", num_layers)
print("Matrices expected:", expected_matrices)
print("Matrices found:", found_matrices)

if found_matrices == expected_matrices:
    print("\nValidation PASSED.")
    print("All target projection matrices were found.")
else:
    print("\nValidation FAILED.")

Transformer blocks: 28
Matrices expected: 196
Matrices found: 196

Validation PASSED.
All target projection matrices were found.


In [36]:
print(
    matrix_inventory["matrix"]
    .value_counts()
    .sort_index()
)

matrix
down_proj    28
gate_proj    28
k_proj       28
o_proj       28
q_proj       28
up_proj      28
v_proj       28
Name: count, dtype: int64


### Save the Matrix Inventory

After validating the inventory, we save it as:

`task01/outputs/matrix_inventory.csv`

CSV format is useful because the inventory can later be opened using Python, Excel, Google Sheets, or other data-analysis tools.

In [37]:
matrix_inventory_path = os.path.join(
    OUTPUT_DIR,
    "matrix_inventory.csv"
)

matrix_inventory.to_csv(
    matrix_inventory_path,
    index=False
)

print("Matrix inventory saved successfully!")
print(matrix_inventory_path)

Matrix inventory saved successfully!
/content/bsr-llm-posttraining/task01/outputs/matrix_inventory.csv


In [38]:
print("Current Task01 output files:\n")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)

    print(
        filename,
        "-",
        os.path.getsize(path),
        "bytes"
    )

Current Task01 output files:

.gitkeep - 0 bytes
matrix_inventory.csv - 17340 bytes
model_structure.txt - 2535 bytes


In [39]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	task01/outputs/matrix_inventory.csv
	task01/outputs/model_structure.txt

nothing added to commit but untracked files present (use "git add" to track)


In [40]:
!git add task01/outputs/model_structure.txt
!git add task01/outputs/matrix_inventory.csv

In [41]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   task01/outputs/matrix_inventory.csv
	new file:   task01/outputs/model_structure.txt



In [42]:
!git commit -m "Add Task01 model structure and matrix inventory"

[main 18577d1] Add Task01 model structure and matrix inventory
 2 files changed, 260 insertions(+)
 create mode 100644 task01/outputs/matrix_inventory.csv
 create mode 100644 task01/outputs/model_structure.txt


In [43]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [44]:
import os
import getpass
import subprocess

GITHUB_USERNAME = "cyenseh-cmyk"

github_token = getpass.getpass(
    "Enter your GitHub Personal Access Token: "
)

askpass_path = "/tmp/git_askpass.sh"

with open(askpass_path, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USERNAME" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = askpass_path
env["GITHUB_USERNAME"] = GITHUB_USERNAME
env["GITHUB_TOKEN"] = github_token
env["GIT_TERMINAL_PROMPT"] = "0"

result = subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

os.remove(askpass_path)
github_token = None

Enter your GitHub Personal Access Token: ··········

To https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git
   0d7cfef..18577d1  main -> main



In [45]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


## Task01 — Step 7: Test Singular Value Decomposition on One Matrix

The next stage of Task01 is spectral analysis.

Before applying spectral analysis to every Transformer projection matrix, we first test Singular Value Decomposition (SVD) on one matrix:

**Transformer Block 0 → `q_proj`**

SVD decomposes a matrix \(W\) into:

\[
W = U \Sigma V^T
\]

where:

- \(U\) contains left singular directions,
- \(\Sigma\) contains the singular values,
- \(V^T\) contains right singular directions.

For Task01, the most important quantities are the **singular values**.

The singular values tell us how strongly the matrix acts along different directions.

Large singular values correspond to strong directions, while smaller singular values correspond to weaker directions.

This one-matrix test helps verify that the SVD calculation works correctly before we repeat it across the full model.

In [46]:
# Select the q_proj matrix from Transformer Block 0
q_proj_weight = model.model.layers[0].self_attn.q_proj.weight

print("Original matrix shape:", q_proj_weight.shape)
print("Original data type:", q_proj_weight.dtype)
print("Original device:", q_proj_weight.device)

# Create a float32 copy for more stable numerical analysis
W = q_proj_weight.detach().float()

print("\nAnalysis matrix shape:", W.shape)
print("Analysis data type:", W.dtype)
print("Analysis device:", W.device)

Original matrix shape: torch.Size([2048, 1024])
Original data type: torch.float16
Original device: cuda:0

Analysis matrix shape: torch.Size([2048, 1024])
Analysis data type: torch.float32
Analysis device: cuda:0


### Step 7.1: Compute the Singular Values

We now compute the singular values of the selected `q_proj` weight matrix.

Because the matrix has shape:

\[
2048 \times 1024
\]

the maximum number of singular values is:

\[
\min(2048,1024)=1024
\]

For this inspection, we print:

- the total number of singular values,
- the ten largest values,
- and the ten smallest values.

This helps us verify that the spectral calculation is working correctly before applying it to the entire model.

In [47]:
with torch.no_grad():
    singular_values = torch.linalg.svdvals(W)

print("Number of singular values:", len(singular_values))

print("\nLargest 10 singular values:")
print(singular_values[:10])

print("\nSmallest 10 singular values:")
print(singular_values[-10:])

Number of singular values: 1024

Largest 10 singular values:
tensor([9.8392, 6.0965, 5.4639, 4.7030, 4.0813, 3.6803, 3.4109, 3.3571, 3.3096,
        3.1104], device='cuda:0')

Smallest 10 singular values:
tensor([0.3485, 0.3451, 0.3384, 0.3351, 0.3306, 0.3295, 0.3259, 0.3159, 0.3065,
        0.2814], device='cuda:0')


### Step 7.2: Validate the Singular Values

Before applying SVD to every projection matrix in Qwen3-0.6B, we verify that the singular values computed for Block 0 `q_proj` are valid.

We check that:

- the number of singular values is correct,
- all singular values are finite,
- all singular values are non-negative,
- and the values are ordered from largest to smallest.

This gives us confidence that the spectral-analysis procedure is working correctly.

In [48]:
expected_count = min(W.shape)

correct_count = len(singular_values) == expected_count
all_finite = torch.isfinite(singular_values).all().item()
all_nonnegative = (singular_values >= 0).all().item()
descending = (
    singular_values[:-1] >= singular_values[1:]
).all().item()

print("Expected singular values:", expected_count)
print("Actual singular values:", len(singular_values))

print("\nValidation checks:")
print("Correct number:", correct_count)
print("All values finite:", all_finite)
print("All values non-negative:", all_nonnegative)
print("Sorted largest to smallest:", descending)

Expected singular values: 1024
Actual singular values: 1024

Validation checks:
Correct number: True
All values finite: True
All values non-negative: True
Sorted largest to smallest: True


### Step 7.3: Calculate Spectral Summary Statistics

Instead of storing all 1,024 singular values for every matrix in one summary table, we calculate several useful statistics.

For each matrix, we can record:

- largest singular value,
- smallest singular value,
- mean singular value,
- median singular value,
- Frobenius norm,
- spectral norm,
- condition number,
- and stable rank.

These values give us a compact description of the matrix's singular-value spectrum.

Later, we will calculate the same statistics for every target projection matrix and save them in:

`task01/outputs/spectral_summary.csv`

In [49]:
sigma_max = singular_values[0].item()
sigma_min = singular_values[-1].item()

mean_sv = singular_values.mean().item()
median_sv = singular_values.median().item()

spectral_norm = sigma_max

frobenius_norm = torch.sqrt(
    torch.sum(singular_values ** 2)
).item()

condition_number = (
    sigma_max / sigma_min
    if sigma_min > 0
    else float("inf")
)

stable_rank = (
    frobenius_norm ** 2
) / (
    spectral_norm ** 2
)

print("Block 0 - q_proj spectral summary")
print("-" * 40)

print("Largest singular value:", sigma_max)
print("Smallest singular value:", sigma_min)
print("Mean singular value:", mean_sv)
print("Median singular value:", median_sv)
print("Spectral norm:", spectral_norm)
print("Frobenius norm:", frobenius_norm)
print("Condition number:", condition_number)
print("Stable rank:", stable_rank)

Block 0 - q_proj spectral summary
----------------------------------------
Largest singular value: 9.839190483093262
Smallest singular value: 0.2814364433288574
Mean singular value: 1.2452480792999268
Median singular value: 1.1506938934326172
Spectral norm: 9.839190483093262
Frobenius norm: 45.39237976074219
Condition number: 34.960612658099166
Stable rank: 21.283701864801493


## Task01 — Step 8: Run Spectral Analysis Across All Target Matrices

The one-matrix test confirmed that Singular Value Decomposition works correctly.

We now repeat the analysis for every target projection matrix in every Transformer block.

For Task01, we only record the **largest singular value** of each matrix.

The largest singular value is:

\[
s_1
\]

It represents the strongest transformation magnitude of the matrix.

The seven matrix types analyzed in each Transformer block are:

- `q_proj`
- `k_proj`
- `v_proj`
- `o_proj`
- `gate_proj`
- `up_proj`
- `down_proj`

For each matrix, the final spectral summary will contain:

- full matrix name,
- Transformer layer,
- matrix family,
- matrix shape,
- largest singular value.

The results will be saved as:

`task01/outputs/spectral_summary.csv`

In [50]:
import torch
import pandas as pd
from tqdm.auto import tqdm

spectral_records = []

num_layers = len(model.model.layers)

for layer_idx, layer in enumerate(
    tqdm(model.model.layers, desc="Analyzing Transformer blocks")
):

    modules = {
        "q_proj": (
            "attention",
            layer.self_attn.q_proj
        ),
        "k_proj": (
            "attention",
            layer.self_attn.k_proj
        ),
        "v_proj": (
            "attention",
            layer.self_attn.v_proj
        ),
        "o_proj": (
            "attention",
            layer.self_attn.o_proj
        ),
        "gate_proj": (
            "mlp",
            layer.mlp.gate_proj
        ),
        "up_proj": (
            "mlp",
            layer.mlp.up_proj
        ),
        "down_proj": (
            "mlp",
            layer.mlp.down_proj
        ),
    }

    for matrix_name, (matrix_family, module) in modules.items():

        # Copy the weight matrix for analysis
        W = module.weight.detach().float()

        # Compute singular values
        with torch.no_grad():
            singular_values = torch.linalg.svdvals(W)

        # Largest singular value
        largest_sv = singular_values[0].item()

        # Build the full matrix name
        if matrix_family == "attention":
            full_name = (
                f"model.layers.{layer_idx}."
                f"self_attn.{matrix_name}.weight"
            )
        else:
            full_name = (
                f"model.layers.{layer_idx}."
                f"mlp.{matrix_name}.weight"
            )

        spectral_records.append({
            "matrix_name": full_name,
            "layer": layer_idx,
            "matrix_family": matrix_family,
            "shape": f"{W.shape[0]}x{W.shape[1]}",
            "largest_singular_value": largest_sv
        })

        # Free temporary tensors
        del W
        del singular_values


spectral_summary = pd.DataFrame(spectral_records)

print("Spectral analysis completed!")
print()
print(spectral_summary.head(14))

Analyzing Transformer blocks:   0%|          | 0/28 [00:00<?, ?it/s]

Spectral analysis completed!

                               matrix_name  layer matrix_family      shape  \
0   model.layers.0.self_attn.q_proj.weight      0     attention  2048x1024   
1   model.layers.0.self_attn.k_proj.weight      0     attention  1024x1024   
2   model.layers.0.self_attn.v_proj.weight      0     attention  1024x1024   
3   model.layers.0.self_attn.o_proj.weight      0     attention  1024x2048   
4      model.layers.0.mlp.gate_proj.weight      0           mlp  3072x1024   
5        model.layers.0.mlp.up_proj.weight      0           mlp  3072x1024   
6      model.layers.0.mlp.down_proj.weight      0           mlp  1024x3072   
7   model.layers.1.self_attn.q_proj.weight      1     attention  2048x1024   
8   model.layers.1.self_attn.k_proj.weight      1     attention  1024x1024   
9   model.layers.1.self_attn.v_proj.weight      1     attention  1024x1024   
10  model.layers.1.self_attn.o_proj.weight      1     attention  1024x2048   
11     model.layers.1.mlp.gate_pro

### Validate the Spectral Summary

Before saving the final spectral results, we verify that:

- every matrix in the inventory received a spectral result,
- no largest singular values are missing,
- no values are NaN,
- all largest singular values are positive.

The number of rows in `spectral_summary.csv` should therefore match the number of rows in `matrix_inventory.csv`.

In [51]:
expected_rows = len(matrix_inventory)
actual_rows = len(spectral_summary)

missing_values = spectral_summary[
    "largest_singular_value"
].isna().sum()

positive_values = (
    spectral_summary["largest_singular_value"] > 0
).all()

print("Matrices in inventory:", expected_rows)
print("Matrices analyzed:", actual_rows)
print("Missing singular values:", missing_values)
print("All largest singular values positive:", positive_values)

if (
    actual_rows == expected_rows
    and missing_values == 0
    and positive_values
):
    print("\nValidation PASSED.")
else:
    print("\nValidation FAILED.")

Matrices in inventory: 196
Matrices analyzed: 196
Missing singular values: 0
All largest singular values positive: True

Validation PASSED.


In [52]:
spectral_summary_path = os.path.join(
    OUTPUT_DIR,
    "spectral_summary.csv"
)

spectral_summary.to_csv(
    spectral_summary_path,
    index=False
)

print("Spectral summary saved successfully!")
print(spectral_summary_path)

Spectral summary saved successfully!
/content/bsr-llm-posttraining/task01/outputs/spectral_summary.csv


In [53]:
print("Final Task01 output files:\n")

for filename in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, filename)

    print(
        filename,
        "-",
        os.path.getsize(path),
        "bytes"
    )

Final Task01 output files:

.gitkeep - 0 bytes
matrix_inventory.csv - 17340 bytes
model_structure.txt - 2535 bytes
spectral_summary.csv - 15036 bytes


In [54]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	task01/outputs/spectral_summary.csv

nothing added to commit but untracked files present (use "git add" to track)


In [55]:
!git add task01/outputs/spectral_summary.csv

In [56]:
!git commit -m "Add Task01 spectral analysis results"

[main f424e64] Add Task01 spectral analysis results
 1 file changed, 197 insertions(+)
 create mode 100644 task01/outputs/spectral_summary.csv


In [57]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [58]:
import os
import getpass
import subprocess

GITHUB_USERNAME = "cyenseh-cmyk"

github_token = getpass.getpass(
    "Enter your GitHub Personal Access Token: "
)

askpass_path = "/tmp/git_askpass.sh"

with open(askpass_path, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USERNAME" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = askpass_path
env["GITHUB_USERNAME"] = GITHUB_USERNAME
env["GITHUB_TOKEN"] = github_token
env["GIT_TERMINAL_PROMPT"] = "0"

result = subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

os.remove(askpass_path)
github_token = None

Enter your GitHub Personal Access Token: ··········

To https://github.com/cyenseh-cmyk/bsr-llm-posttraining.git
   18577d1..f424e64  main -> main



In [59]:
!git status


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [60]:
import os
import pandas as pd

required_files = [
    "model_structure.txt",
    "matrix_inventory.csv",
    "spectral_summary.csv"
]

print("TASK01 OUTPUT CHECK")
print("=" * 50)

for filename in required_files:
    path = os.path.join(OUTPUT_DIR, filename)

    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0

    print(f"{filename}")
    print(f"  Exists: {exists}")
    print(f"  Size: {size} bytes")

TASK01 OUTPUT CHECK
model_structure.txt
  Exists: True
  Size: 2535 bytes
matrix_inventory.csv
  Exists: True
  Size: 17340 bytes
spectral_summary.csv
  Exists: True
  Size: 15036 bytes


In [61]:
inventory_check = pd.read_csv(
    os.path.join(OUTPUT_DIR, "matrix_inventory.csv")
)

spectral_check = pd.read_csv(
    os.path.join(OUTPUT_DIR, "spectral_summary.csv")
)

print("Matrices in inventory:", len(inventory_check))
print("Matrices in spectral summary:", len(spectral_check))

if len(inventory_check) == len(spectral_check):
    print("\nPASS: Every inventoried matrix has a spectral result.")
else:
    print("\nFAIL: The row counts do not match.")

Matrices in inventory: 196
Matrices in spectral summary: 196

PASS: Every inventoried matrix has a spectral result.


In [62]:
!find task01 -maxdepth 2 -type f -print | sort

task01/outputs/.gitkeep
task01/outputs/matrix_inventory.csv
task01/outputs/model_structure.txt
task01/outputs/spectral_summary.csv


In [64]:
!git ls-files | grep -E '\.(ipynb|py)$'